# Module 7 — Monitoring avec Evidently AI

Ce notebook implémente le monitoring en production :
1. **Data drift numérique** : détection via Evidently AI
2. **Text drift** : divergence Jensen-Shannon sur les distributions TF-IDF
3. **Alertes** sur la performance (seuil accuracy ≥ 0.70)

**Prérequis** : `pip install evidently`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial.distance import jensenshannon
from sklearn.feature_extraction.text import TfidfVectorizer
import json, logging, datetime

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger('monitoring')

DATA_DIR = Path('..') / 'data' / 'processed'
ref_df  = pd.read_csv(DATA_DIR / 'train_clean.csv')
curr_df = pd.read_csv(DATA_DIR / 'test_clean.csv')

print('Référence:', ref_df.shape, '| Courant:', curr_df.shape)

## 1. Data Drift Numérique (Evidently AI)

In [ ]:
try:
    from evidently.report import Report
    from evidently.metric_preset import DataDriftPreset, DataQualityPreset
    from evidently.metrics import DatasetDriftMetric, ColumnDriftMetric

    num_cols = ['Poids', 'Volume', 'Conductivite', 'Opacite', 'Rigidite']
    ref_num = ref_df[num_cols].copy()
    curr_num = curr_df[num_cols].copy()

    report = Report(metrics=[
        DataDriftPreset(),
        DataQualityPreset(),
    ])
    report.run(reference_data=ref_num, current_data=curr_num)
    report.save_html('drift_report.html')
    print('✅ Rapport Evidently sauvegardé : drift_report.html')

    # Extraire les métriques de drift
    result = report.as_dict()
    drift_detected = result['metrics'][0]['result'].get('dataset_drift', False)
    print(f'Dataset drift détecté : {drift_detected}')

except ImportError:
    print('⚠️  Evidently non installé. Installe avec: pip install evidently')
    print('Fallback : calcul manuel du drift via statistiques de base')
    num_cols = ['Poids', 'Volume', 'Conductivite', 'Opacite', 'Rigidite']
    for col in num_cols:
        ref_mean = ref_df[col].mean()
        cur_mean = curr_df[col].mean()
        drift_pct = abs(ref_mean - cur_mean) / (abs(ref_mean) + 1e-8) * 100
        status = '⚠️  DRIFT' if drift_pct > 10 else '✅ OK'
        print(f'{col:<20} ref_mean={ref_mean:.3f}  cur_mean={cur_mean:.3f}  Δ={drift_pct:.1f}%  {status}')

## 2. Text Drift — Divergence Jensen-Shannon

In [ ]:
TEXT_COL = 'Rapport_Collecte'

if TEXT_COL in ref_df.columns and TEXT_COL in curr_df.columns:
    ref_texts  = ref_df[TEXT_COL].fillna('').tolist()
    curr_texts = curr_df[TEXT_COL].fillna('').tolist()

    # Vectorisation TF-IDF partagée
    vectorizer = TfidfVectorizer(max_features=500, ngram_range=(1, 2))
    vectorizer.fit(ref_texts + curr_texts)

    ref_tfidf  = np.asarray(vectorizer.transform(ref_texts).mean(axis=0)).flatten()
    curr_tfidf = np.asarray(vectorizer.transform(curr_texts).mean(axis=0)).flatten()

    # Normalisation pour en faire des distributions de probabilité
    ref_prob  = ref_tfidf  / (ref_tfidf.sum()  + 1e-10)
    curr_prob = curr_tfidf / (curr_tfidf.sum() + 1e-10)

    js_distance = jensenshannon(ref_prob, curr_prob)
    print(f'Jensen-Shannon Distance (texte): {js_distance:.4f}')
    if js_distance > 0.1:
        print('⚠️  ALERTE : Dérive textuelle significative détectée !')
    else:
        print('✅ Pas de dérive textuelle significative.')

## 3. Alerte Performance + Logging JSON

In [ ]:
import joblib
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

MODEL_PATH = Path('..') / 'notebooks' / 'multimodal_model.pkl'
PERF_THRESHOLD = 0.70
LOG_FILE = 'monitoring_log.jsonl'

if not MODEL_PATH.exists():
    print('⚠️  Modèle PKL non trouvé. Lance le notebook 06 pour le générer.')
else:
    model = joblib.load(MODEL_PATH)
    text_col = 'Rapport_Collecte'
    num_cols = [c for c in curr_df.select_dtypes(include=['number']).columns
                if c not in ['Prix_Revente']]

    curr_df[text_col] = curr_df[text_col].fillna('')
    X_curr = curr_df[[text_col] + num_cols]
    y_curr = curr_df['Categorie']

    preds = model.predict(X_curr)
    acc = accuracy_score(y_curr, preds)

    # Logging JSON structuré
    log_entry = {
        'timestamp': datetime.datetime.utcnow().isoformat(),
        'accuracy': round(acc, 4),
        'threshold': PERF_THRESHOLD,
        'alert': acc < PERF_THRESHOLD,
        'n_samples': len(y_curr),
    }
    with open(LOG_FILE, 'a') as f:
        f.write(json.dumps(log_entry) + '\n')

    print(f'Accuracy courante : {acc:.4f}')
    if acc < PERF_THRESHOLD:
        print(f'🚨 ALERTE PERFORMANCE : accuracy {acc:.4f} < seuil {PERF_THRESHOLD}')
        logger.warning(f'Performance dégradée: accuracy={acc:.4f}')
    else:
        print(f'✅ Performance OK : accuracy {acc:.4f} ≥ {PERF_THRESHOLD}')
    print(f'Log écrit dans {LOG_FILE}')